# Stage 1 – 1‑Week Cap vs 1‑Month Cap Diagnostics

Compares the seasonal runs:
- `stagel_3month_1week_cap`
- `stagel_3month_1month_cap`


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")


In [ ]:
# --- User configuration -----------------------------------------------------
PROJECT_DIR = Path("/home/fs01/jl2966/acorn-julia2/acorn-julia")
RUN_NAME = "low_RE_mod_elec_iter0"
CLIMATE_SCENARIO = "historical_1980_2019"

RUNS = {
    "week_cap": "stagel_3month_1week_cap",
    "month_cap": "stagel_3month",
}

YEAR = 1985

output_root = PROJECT_DIR / "runs" / RUN_NAME / "outputs" / CLIMATE_SCENARIO
print("Runs:", RUNS)
print("Runs:", RUNS)
print("Year:", YEAR)


In [ ]:
# --- Helpers ----------------------------------------------------------------

def tidy_storage_df(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    meta_cols = [c for c in ["bus_id", "asset_type", "zone", "is_seasonal"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    return tidy


def tidy_bus_csv(path: Path, value_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    meta_cols = [c for c in ["bus_id", "zone"] if c in df.columns]
    value_cols = [c for c in df.columns if c not in meta_cols]
    tidy = df.melt(id_vars=meta_cols, value_vars=value_cols,
                   var_name="timestamp", value_name=value_name)
    tidy["timestamp"] = pd.to_datetime(tidy["timestamp"], errors="coerce")
    tidy = tidy.dropna(subset=["timestamp"])
    return tidy


def load_storage_csv(run_dir: Path, year: int, value_name: str, split: str | None = None,
                     fallback_combined: bool = True) -> pd.DataFrame:
    if split in {"base", "seasonal"}:
        split_path = run_dir / f"{value_name}_{split}_{year}.csv"
        if split_path.exists():
            df = pd.read_csv(split_path)
            df = df.copy()
            df["asset_type"] = split
            df["is_seasonal"] = 1 if split == "seasonal" else 0
            return tidy_storage_df(df, value_name)
        if not fallback_combined:
            raise FileNotFoundError(split_path)

    df = pd.read_csv(run_dir / f"{value_name}_{year}.csv")
    debug_path = run_dir / "_storage_debug.csv"
    if debug_path.exists():
        debug = pd.read_csv(debug_path)
        if "is_seasonal" in debug.columns and len(debug) == len(df):
            df = df.copy()
            if "asset_type" not in df.columns:
                df["asset_type"] = "base"
            df.loc[debug["is_seasonal"] == 1, "asset_type"] = "seasonal"
            df["is_seasonal"] = debug["is_seasonal"].values

    if split in {"base", "seasonal"} and "asset_type" in df.columns:
        df = df[df["asset_type"] == split].copy()

    return tidy_storage_df(df, value_name)


def total_timeseries(df: pd.DataFrame, value_name: str) -> pd.Series:
    return df.groupby("timestamp")[value_name].sum()


def event_stats(series: pd.Series, threshold: float = 1e-6) -> pd.DataFrame:
    s = series.fillna(0.0)
    events = []
    in_event = False
    start = None
    total = 0.0
    hours = 0
    for ts, val in s.items():
        if val > threshold:
            if not in_event:
                in_event = True
                start = ts
                total = 0.0
                hours = 0
            total += float(val)
            hours += 1
            end = ts
        else:
            if in_event:
                events.append((start, end, hours, total))
                in_event = False
    if in_event:
        events.append((start, end, hours, total))

    if not events:
        return pd.DataFrame(columns=["start", "end", "hours", "total_MWh"])
    df = pd.DataFrame(events, columns=["start", "end", "hours", "total_MWh"])
    return df.sort_values("hours", ascending=False)


In [ ]:
# --- Load runs --------------------------------------------------------------
run_data = {label: {} for label in RUNS}

for label, save_name in RUNS.items():
    run_dir = output_root / save_name
    charge_base = load_storage_csv(run_dir, YEAR, "charge", split="base")
    discharge_base = load_storage_csv(run_dir, YEAR, "discharge", split="base")
    charge_seasonal = load_storage_csv(run_dir, YEAR, "charge", split="seasonal")
    discharge_seasonal = load_storage_csv(run_dir, YEAR, "discharge", split="seasonal")
    load_shed = tidy_bus_csv(run_dir / f"load_shedding_{YEAR}.csv", "load_shedding")

    run_data[label][YEAR] = {
        "charge_base": charge_base,
        "discharge_base": discharge_base,
        "charge_seasonal": charge_seasonal,
        "discharge_seasonal": discharge_seasonal,
        "load_shed": load_shed,
    }

print("Loaded:", {k: list(v.keys()) for k, v in run_data.items()})


## Seasonal Usage Summary

In [ ]:
# --- Numeric comparison (no graphs) ----------------------------------------
rows = []
for label in RUNS:
    run = run_data[label][YEAR]
    base_dis = total_timeseries(run["discharge_base"], "discharge")
    seasonal_dis = total_timeseries(run["discharge_seasonal"], "discharge")
    base_ch = total_timeseries(run["charge_base"], "charge")
    seasonal_ch = total_timeseries(run["charge_seasonal"], "charge")
    ls = total_timeseries(run["load_shed"], "load_shedding")

    rows.append({
        "run": label,
        "base_discharge_MWh": base_dis.sum(),
        "seasonal_discharge_MWh": seasonal_dis.sum(),
        "base_charge_MWh": base_ch.sum(),
        "seasonal_charge_MWh": seasonal_ch.sum(),
        "seasonal_discharge_hours": int((seasonal_dis > 0).sum()),
        "seasonal_charge_hours": int((seasonal_ch > 0).sum()),
        "load_shed_MWh": ls.sum(),
        "load_shed_hours": int((ls > 0).sum()),
        "peak_load_shed_MW": float(ls.max()),
    })

summary = pd.DataFrame(rows)
summary["seasonal_discharge_MWh"] = summary["seasonal_discharge_MWh"].round(2)
summary["seasonal_charge_MWh"] = summary["seasonal_charge_MWh"].round(2)
summary["base_discharge_MWh"] = summary["base_discharge_MWh"].round(2)
summary["base_charge_MWh"] = summary["base_charge_MWh"].round(2)
summary["load_shed_MWh"] = summary["load_shed_MWh"].round(2)
summary["peak_load_shed_MW"] = summary["peak_load_shed_MW"].round(2)

display(summary)


In [ ]:
rows = []
for label in RUNS:
    run = run_data[label][YEAR]
    base_dis = total_timeseries(run["discharge_base"], "discharge")
    seasonal_dis = total_timeseries(run["discharge_seasonal"], "discharge")
    ls = total_timeseries(run["load_shed"], "load_shedding")
    rows.append({
        "run": label,
        "base_discharge_MWh": base_dis.sum(),
        "seasonal_discharge_MWh": seasonal_dis.sum(),
        "seasonal_discharge_hours": int((seasonal_dis > 0).sum()),
        "load_shed_MWh": ls.sum(),
        "load_shed_hours": int((ls > 0).sum()),
    })

display(pd.DataFrame(rows))


## Fleet Activity (Base vs Seasonal)

In [ ]:
# Week cap vs month cap — separate charts per run
colors = {
    "base_charge": "#1f77b4",
    "base_discharge": "#1f77b4",
    "seasonal_charge": "#ff7f0e",
    "seasonal_discharge": "#ff7f0e",
}

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# --- Week cap chart
label = "week_cap"
run = run_data[label][YEAR]
base_ch = total_timeseries(run["charge_base"], "charge")
base_dis = total_timeseries(run["discharge_base"], "discharge")
seasonal_ch = total_timeseries(run["charge_seasonal"], "charge")
seasonal_dis = total_timeseries(run["discharge_seasonal"], "discharge")

axes[0].plot(base_ch.index, base_ch.values, label="base charge", color=colors["base_charge"], alpha=0.6)
axes[0].plot(base_dis.index, -base_dis.values, label="base discharge", color=colors["base_discharge"], alpha=0.9, linestyle="--")
axes[0].plot(seasonal_ch.index, seasonal_ch.values, label="seasonal charge", color=colors["seasonal_charge"], alpha=0.6)
axes[0].plot(seasonal_dis.index, -seasonal_dis.values, label="seasonal discharge", color=colors["seasonal_discharge"], alpha=0.9, linestyle=":")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Week cap: base vs seasonal charge/discharge")
axes[0].legend(ncol=2, fontsize=8)

# --- Month cap chart
label = "month_cap"
run = run_data[label][YEAR]
base_ch = total_timeseries(run["charge_base"], "charge")
base_dis = total_timeseries(run["discharge_base"], "discharge")
seasonal_ch = total_timeseries(run["charge_seasonal"], "charge")
seasonal_dis = total_timeseries(run["discharge_seasonal"], "discharge")

axes[1].plot(base_ch.index, base_ch.values, label="base charge", color=colors["base_charge"], alpha=0.6)
axes[1].plot(base_dis.index, -base_dis.values, label="base discharge", color=colors["base_discharge"], alpha=0.9, linestyle="--")
axes[1].plot(seasonal_ch.index, seasonal_ch.values, label="seasonal charge", color=colors["seasonal_charge"], alpha=0.6)
axes[1].plot(seasonal_dis.index, -seasonal_dis.values, label="seasonal discharge", color=colors["seasonal_discharge"], alpha=0.9, linestyle=":")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Month cap: base vs seasonal charge/discharge")
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()


## Long Shortage Events

In [ ]:
for label in RUNS:
    ls = total_timeseries(run_data[label][YEAR]["load_shed"], "load_shedding")
    events = event_stats(ls, threshold=0.0)
    print(f"{label} – top load shedding events")
    display(events.head(10))
